# Install packages

In [1]:
!pip install geopandas

DEPRECATION: Loading egg at c:\users\mchlc\appdata\local\programs\python\python312\lib\site-packages\dcnet-0.0.1-py3.12.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..


In [2]:
import geopandas as gpd
import pandas as pd

c:\Users\mchlc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\mchlc\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


# Data processing

### [COMMERICAL FISHERIES] This reduces fisheries data size by aggregating by polygons.

Input: `../public/fisheriesdata/20260216_tidied_comm_ev.gpkg` (commercial fisheries data from Alemarie - removed to save space.)

Output: `../public/fisheriesdata/FOR_PROJECT/09182026_comm_EV_byCatchmentArea.geojson` (geoJSON, ready to be used)

In [ ]:
geojson_df = gpd.read_file("../public/fisheriesdata/20260216_tidied_comm_ev.gpkg")

crs = geojson_df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    
    return list(group_data)

geojson_df.drop(["county", "exchange_value_formatted"], inplace=True, axis=1)
geojson_df["area_id"] = geojson_df["area_id"].apply(lambda x: "Catchment Area " + str(x))
geojson_df["exchange_value"] = geojson_df["exchange_value"].fillna(-1.0)
x = geojson_df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

# check if it's all good.

new_gdf = gpd.GeoDataFrame(x, geometry="geometry", crs=crs)
#new_gdf.to_file("collapsed_comm_ev.geojson", driver="geoJSON")

new_gdf.to_file("../public/fisheriesdata/FOR_PROJECT/10_14_2026_comm_EV_byCatchmentArea.geojson", driver="geoJSON", COORDINATE_PRECISION=5)
x.head(2)

In [ ]:
def is_valid_df(x):
    must_have = {"geometry", "year", "area_id", "county_olelo", "species_group", "ecosystem_type", "exchange_value"}
    current_columns = set(x.columns)

    if (must_have.issubset(current_columns)):
        print("✅ All columns are valid!")
    else:
        print("❌ The following columns are missing: " + str(must_have - current_columns))
        print("We require the following columns: " + str(must_have))
        return False;

    are_rows_valid = True;
    for row in x.index:
        if ((len(x.loc[row, "year"]) == len(x.loc[row, "species_group"])) and (len(x.loc[row, "species_group"]) == len(x.loc[row, "ecosystem_type"])) and (len(x.loc[row, "ecosystem_type"]) == len(x.loc[row, "exchange_value"]))):
            pass
        else:
            print("❌ Row " + str(row) + " is invalid!")
            cols_to_compare = ["year", "species_group", "ecosystem_type", "exchange_value"]
            for col in cols_to_compare:
                print("Length of '" + col + "' is: " + str(len(x.loc[row, col])))
            print("Array lengths mismatch.")
            print("=====")
            are_rows_valid = False;

    if (are_rows_valid):
        print("✅ All rows are valid!")
    else:
        print("❌ Some columns are invalid. See above.")

    return are_rows_valid;

is_valid_df(x)

❌ The following columns are missing: {'hargga'}
We require the following columns: {'year', 'county_olelo', 'geometry', 'area_id', 'exchange_value', 'species_group', 'ecosystem_type', 'hargga'}


False

### [NONCOMMERCIAL FISHERIES] Apply the same collapsing.

Input: `../public/fisheriesdata/20260216_tidied_noncomm_ev.gpkg` (noncommercial fisheries data from Alemarie)

Output: `../public/fisheriesdata/USED_IN_PROJECT/10_14_2026_comm_EV_byIsland.geojson` (geoJSON, ready to be used)

In [12]:
geojson_df = gpd.read_file("../public/fisheriesdata/20260216_tidied_noncomm_ev.gpkg")

crs = geojson_df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    
    return list(group_data)

geojson_df.drop(["island", "county", "exchange_value_formatted"], inplace=True, axis=1)
geojson_df["exchange_value"] = geojson_df["exchange_value"].fillna(-1.0)
x = geojson_df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

# check if it's all good.

new_gdf = gpd.GeoDataFrame(x, geometry="geometry", crs=crs)
#new_gdf.to_file("collapsed_comm_ev.geojson", driver="geoJSON")

new_gdf.to_file("../public/fisheriesdata/USED_IN_PROJECT/10_14_2026_comm_EV_byIsland.geojson", driver="geoJSON", COORDINATE_PRECISION=5)
x

,geometry,year,island_olelo,county_olelo,species_group,ecosystem_type,exchange_value
0,"POLYGON ((-159.61896 22.25362, -159.62655 22.2...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Kauaʻi,Kauaʻi,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[73724.19430652258, 73724.19430652258, 0.0, 11..."
1,"POLYGON ((-158.11157 21.65975, -158.1205 21.65...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Oʻahu,Honolulu,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[316375.55467450985, 316375.55467450985, 0.0, ..."
2,"POLYGON ((-157.07334 20.95462, -157.07481 20.9...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Lānaʻi,Maui,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[8315.04691017696, 8315.04691017696, 0.0, 1325..."
3,"POLYGON ((-157.22929 21.26431, -157.23074 21.2...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Molokaʻi,Maui,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[71245.35504390174, 71245.35504390174, 0.0, 11..."
4,"POLYGON ((-156.74179 20.94513, -156.74281 20.9...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Maui,Maui,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[131057.17653438573, 131057.17653438573, 0.0, ..."
5,"POLYGON ((-155.68387 20.23114, -155.68972 20.2...","[2005, 2005, 2005, 2006, 2006, 2006, 2007, 200...",Hawaiʻi,Hawaiʻi,Herbivores,"[All Ecosystems, Inshore — Reef, Coastal — Ope...","[300588.77101082925, 300588.77101082925, 0.0, ..."


### [TERRESTRIAL MOKU EXTENTS] Merge the moku extents data, also by aggregating by polygon.

Input: `../public/mokuextentsdata/mokus_extents.geojson` (from who?)

Output: `../public/mokuextentsdata/USED_IN_PROJECT/2026_10_07_terrestrial_extents_byMoku.geojson` (geoJSON, ready to be used)

In [ ]:
df = gpd.read_file("../public/mokuextentsdata/mokus_extents.geojson")

crs = df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    if (len(set(group_data)) == 0):
        return []
    
    final_list = list(group_data)
    return [x if not x is None else 0 for x in final_list]

df.drop(["fid", "name2", "moku", "island"], inplace=True, axis=1)
x = df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

#marine_df = x.loc[x["realm"] == "Marine"]
terrestrial_df = x.loc[x["realm"] == "Terrestrial"]

terrestrial_df["total_area_km2"] = terrestrial_df.apply(lambda x: sum(x["area_km2"]) / len(set(x["year"])) if len(x["area_km2"]) > 0 else 0, axis=1)

#new_marine = gpd.GeoDataFrame(marine_df, geometry="geometry", crs=crs)
new_terrestrial = gpd.GeoDataFrame(terrestrial_df, geometry="geometry", crs=crs)

new_terrestrial.to_file("../public/mokuextentsdata/USED_IN_PROJECT/2026_10_07_terrestrial_extents_byMoku.geojson", driver="geoJSON", COORDINATE_PRECISION=5)